# 谁动了我的水资源？——归因分析
# Who Took My Water? - Attribution Analysis

---

## 🌍 一个真实的问题 | A Real-World Question

**中文场景**：

黄河流域的年径流量从 1980 年代开始显著减少。科学家们争论：
- 🌡️ **气候派**：是全球变暖导致蒸发增加！
- 🌲 **土地派**：是植树造林和农业灌溉消耗了水！

**到底谁说得对？我们需要**归因分析**！**

---

**English Scenario**:

The annual runoff of the Yellow River Basin has significantly decreased since the 1980s. Scientists debate:
- 🌡️ **Climate faction**: It's global warming causing increased evaporation!
- 🌲 **Land faction**: It's afforestation and agricultural irrigation consuming water!

**Who is right? We need attribution analysis!**

---

## 🧮 Budyko 框架：水-能量平衡 | Budyko Framework: Water-Energy Balance

### 核心思想 | Core Idea:

俄罗斯科学家 Budyko (1974) 发现，**长期平均蒸散发**取决于两个关键比率：

Russian scientist Budyko (1974) found that **long-term average ET** depends on two key ratios:

1. **干燥度指数 | Aridity Index** ($\phi$):
$$
\phi = \frac{PET_e}{P}
$$
   - $PET_e$: 能量限制的潜在蒸散发 | Energy-limited potential ET
   - $P$: 降水 | Precipitation
   - **物理意义**：大气需求 vs 水分供应
   - **Physical meaning**: Atmospheric demand vs water supply

2. **蒸发比 | Evaporative Index** ($\varepsilon$):
$$
\varepsilon = \frac{ET}{P}
$$
   - $ET$: 实际蒸散发 | Actual ET
   - **物理意义**：有多少降水被蒸发了？
   - **Physical meaning**: How much precipitation is evaporated?

### Budyko 曲线 | Budyko Curve:

$$
\varepsilon = f(\phi, n)
$$

其中 $n$ 是**流域参数**（反映下垫面特征，如植被、土壤、地形）。

Where $n$ is the **catchment parameter** (reflecting surface characteristics like vegetation, soil, topography).

---

## 🎯 归因分析的逻辑 | Logic of Attribution Analysis

**关键洞察 | Key Insight**:

如果 $\Delta ET$ (ET 的变化) 是由于：

If $\Delta ET$ (change in ET) is due to:

1. **气候变化** → $\phi$ 改变（降水或 PET 变化）→ 在 Budyko 曲线上**沿曲线移动**
   - **Climate change** → $\phi$ changes (precipitation or PET varies) → **move along the Budyko curve**

2. **下垫面变化** → $n$ 改变（植被增加、城市化等）→ **曲线形状改变**
   - **Land surface change** → $n$ changes (vegetation increase, urbanization, etc.) → **curve shape changes**

**数学分解 | Mathematical Decomposition**:

$$
\Delta ET = \underbrace{\frac{\partial ET}{\partial \phi} \Delta \phi}_{\text{气候贡献 | Climate}} + \underbrace{\frac{\partial ET}{\partial n} \Delta n}_{\text{下垫面贡献 | Land surface}}
$$

---

## 💻 代码实现 | Code Implementation

In [ ]:
# 导入库 | Import libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import sys
sys.path.insert(0, '..')
import petcr

print("✅ 库导入成功！Libraries imported successfully!")

## 🧪 实验：模拟 30 年的水文变化 | Experiment: Simulate 30 Years of Hydrological Change

In [ ]:
# 生成模拟数据（30年时间序列）
# Generate simulated data (30-year time series)
np.random.seed(42)

n_years = 30
years = np.arange(1990, 1990 + n_years)

# 场景 1: 基线期（1990-1999）
# Scenario 1: Baseline period (1990-1999)
baseline_years = 10

# 场景 2: 气候变化期（2000-2019）
# Scenario 2: Climate change period (2000-2019)
# - 降水减少 10%
# - PETe 增加 15%（气温升高）
# - 植被增加导致 n 从 2.5 → 3.0

# 基线气候条件 | Baseline climate
precip_baseline = 800.0  # mm/year
pete_baseline = 1200.0   # mm/year
n_baseline = 2.5         # Budyko parameter

# 构建时间序列 | Construct time series
precip = np.zeros(n_years)
pete = np.zeros(n_years)
n_param = np.zeros(n_years)

for i in range(n_years):
    if i < baseline_years:
        # 基线期 | Baseline
        precip[i] = precip_baseline + np.random.normal(0, 50)
        pete[i] = pete_baseline + np.random.normal(0, 80)
        n_param[i] = n_baseline
    else:
        # 变化期 | Change period
        # 线性趋势 | Linear trend
        year_offset = i - baseline_years
        precip_trend = -10 * year_offset / 20  # 降水逐渐减少
        pete_trend = 15 * year_offset / 20     # PETe 逐渐增加
        n_trend = 0.5 * year_offset / 20       # n 逐渐增加（植被增加）
        
        precip[i] = precip_baseline + precip_trend + np.random.normal(0, 50)
        pete[i] = pete_baseline + pete_trend + np.random.normal(0, 80)
        n_param[i] = n_baseline + n_trend

# 确保物理合理性 | Ensure physical plausibility
precip = np.clip(precip, 400, 1200)
pete = np.clip(pete, 800, 1600)
n_param = np.clip(n_param, 2.0, 3.5)

print(f"数据生成完成 | Data generation complete")
print(f"年份范围 | Year range: {years[0]} - {years[-1]}")
print(f"降水范围 | Precipitation range: {precip.min():.1f} - {precip.max():.1f} mm/year")
print(f"PETe 范围 | PETe range: {pete.min():.1f} - {pete.max():.1f} mm/year")

## 📊 计算实际蒸散发和归因 | Calculate Actual ET and Attribution

In [ ]:
# 定义 Budyko 函数 | Define Budyko function
def budyko_curve(phi, n):
    """
    Budyko 曲线（Fu 1981 参数化）
    Budyko curve (Fu 1981 parameterization)
    
    Parameters:
    - phi: 干燥度指数 (PETe/P) | Aridity index (PETe/P)
    - n: 流域参数 | Catchment parameter
    
    Returns:
    - epsilon: 蒸发比 (ET/P) | Evaporative index (ET/P)
    """
    epsilon = 1 + phi - (1 + phi**n)**(1/n)
    return np.clip(epsilon, 0, 1)  # 物理约束：0 <= epsilon <= 1

# 计算每年的 ET | Calculate annual ET
et_annual = np.zeros(n_years)
phi_annual = np.zeros(n_years)
epsilon_annual = np.zeros(n_years)

for i in range(n_years):
    phi = pete[i] / precip[i]
    epsilon = budyko_curve(phi, n_param[i])
    et = epsilon * precip[i]
    
    phi_annual[i] = phi
    epsilon_annual[i] = epsilon
    et_annual[i] = et

print(f"\n✅ ET 计算完成 | ET calculation complete")
print(f"基线期平均 ET | Baseline ET: {et_annual[:baseline_years].mean():.1f} mm/year")
print(f"变化期平均 ET | Change period ET: {et_annual[baseline_years:].mean():.1f} mm/year")
print(f"ET 变化 | ET change: {et_annual[baseline_years:].mean() - et_annual[:baseline_years].mean():.1f} mm/year")

## 🔍 归因分析：分离气候和下垫面影响 | Attribution: Separate Climate and Land Surface Effects

In [ ]:
# 归因分析方法：使用微分法
# Attribution method: Differential approach

# 定义参考期和对比期 | Define reference and comparison periods
ref_period = slice(0, baseline_years)  # 1990-1999
comp_period = slice(baseline_years, n_years)  # 2000-2019

# 计算平均值 | Calculate averages
precip_ref = precip[ref_period].mean()
pete_ref = pete[ref_period].mean()
n_ref = n_param[ref_period].mean()
et_ref = et_annual[ref_period].mean()

precip_comp = precip[comp_period].mean()
pete_comp = pete[comp_period].mean()
n_comp = n_param[comp_period].mean()
et_comp = et_annual[comp_period].mean()

# 计算变化量 | Calculate changes
delta_et_total = et_comp - et_ref
delta_precip = precip_comp - precip_ref
delta_pete = pete_comp - pete_ref
delta_n = n_comp - n_ref

# 归因计算（简化方法）| Attribution calculation (simplified)
# 方法 1: 固定 n，改变气候 | Method 1: Fix n, change climate
phi_ref = pete_ref / precip_ref
phi_comp_climate = pete_comp / precip_comp  # 仅气候变化

epsilon_ref = budyko_curve(phi_ref, n_ref)
epsilon_climate_only = budyko_curve(phi_comp_climate, n_ref)  # 固定 n
et_climate_only = epsilon_climate_only * precip_comp

delta_et_climate = et_climate_only - et_ref

# 方法 2: 固定气候，改变 n | Method 2: Fix climate, change n
epsilon_landsurf_only = budyko_curve(phi_comp_climate, n_comp)  # 改变 n
et_landsurf_only = epsilon_landsurf_only * precip_comp

delta_et_landsurf = et_landsurf_only - et_climate_only

# 验证守恒 | Verify conservation
delta_et_reconstructed = delta_et_climate + delta_et_landsurf

print("\n" + "═" * 70)
print("🔍 归因分析结果 | Attribution Analysis Results")
print("═" * 70)
print(f"\n📊 总变化 | Total Change:")
print(f"   实际 ET 变化 | Actual ET change:    {delta_et_total:.2f} mm/year")
print(f"   重构 ET 变化 | Reconstructed:       {delta_et_reconstructed:.2f} mm/year")
print(f"   误差 | Error:                      {abs(delta_et_total - delta_et_reconstructed):.2f} mm/year")

print(f"\n🌡️ 气候贡献 | Climate Contribution:")
print(f"   ΔET (气候) | Climate:               {delta_et_climate:.2f} mm/year")
print(f"   占比 | Percentage:                  {(delta_et_climate/delta_et_total*100):.1f}%")

print(f"\n🌲 下垫面贡献 | Land Surface Contribution:")
print(f"   ΔET (下垫面) | Land surface:       {delta_et_landsurf:.2f} mm/year")
print(f"   占比 | Percentage:                  {(delta_et_landsurf/delta_et_total*100):.1f}%")

print(f"\n📈 驱动因子变化 | Driving Factor Changes:")
print(f"   降水变化 | Precipitation change:   {delta_precip:.1f} mm/year ({delta_precip/precip_ref*100:.1f}%)")
print(f"   PETe 变化 | PETe change:            {delta_pete:.1f} mm/year ({delta_pete/pete_ref*100:.1f}%)")
print(f"   n 参数变化 | n parameter change:   {delta_n:.3f} ({delta_n/n_ref*100:.1f}%)")
print("═" * 70)

## 📈 可视化：时间序列和 Budyko 图 | Visualization: Time Series and Budyko Plot

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. 降水和 PETe 时间序列 | Precipitation and PETe time series
ax1 = fig.add_subplot(gs[0, :])
ax1_twin = ax1.twinx()

line1 = ax1.plot(years, precip, 'b-o', linewidth=2, markersize=4, label='降水 P | Precipitation')
line2 = ax1_twin.plot(years, pete, 'r-s', linewidth=2, markersize=4, label='PETe')

ax1.axvline(x=years[baseline_years], color='gray', linestyle='--', linewidth=2, alpha=0.7)
ax1.text(years[baseline_years/2], ax1.get_ylim()[1]*0.95, '基线期\nBaseline', 
         ha='center', fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
ax1.text(years[baseline_years + (n_years-baseline_years)/2], ax1.get_ylim()[1]*0.95, 
         '变化期\nChange', ha='center', fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

ax1.set_xlabel('年份 | Year', fontweight='bold', fontsize=12)
ax1.set_ylabel('降水 (mm/year) | Precipitation (mm/year)', color='b', fontweight='bold', fontsize=11)
ax1_twin.set_ylabel('PETe (mm/year)', color='r', fontweight='bold', fontsize=11)
ax1.tick_params(axis='y', labelcolor='b')
ax1_twin.tick_params(axis='y', labelcolor='r')
ax1.set_title('气候驱动因子时间序列 | Climate Driving Factors Time Series', 
              fontweight='bold', fontsize=13)
ax1.grid(alpha=0.3)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=11)

# 2. ET 时间序列 | ET time series
ax2 = fig.add_subplot(gs[1, :])
ax2.plot(years, et_annual, 'g-D', linewidth=2.5, markersize=5, label='实际 ET | Actual ET')
ax2.fill_between(years, et_annual, alpha=0.3, color='green')
ax2.axhline(y=et_ref, color='blue', linestyle=':', linewidth=2, label='基线期平均 | Baseline mean')
ax2.axhline(y=et_comp, color='red', linestyle=':', linewidth=2, label='变化期平均 | Change period mean')
ax2.axvline(x=years[baseline_years], color='gray', linestyle='--', linewidth=2, alpha=0.7)

ax2.set_xlabel('年份 | Year', fontweight='bold', fontsize=12)
ax2.set_ylabel('实际蒸散发 (mm/year) | Actual ET (mm/year)', fontweight='bold', fontsize=12)
ax2.set_title('实际蒸散发时间序列 | Actual ET Time Series', fontweight='bold', fontsize=13)
ax2.legend(fontsize=11, loc='best')
ax2.grid(alpha=0.3)

# 添加变化量标注 | Add change annotation
ax2.annotate('', xy=(years[-5], et_ref), xytext=(years[-5], et_comp),
            arrowprops=dict(arrowstyle='<->', lw=2, color='purple'))
ax2.text(years[-5]+1, (et_ref + et_comp)/2, 
         f'ΔET = {delta_et_total:.1f} mm/year',
         fontsize=11, fontweight='bold', color='purple',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 3. Budyko 曲线 | Budyko curve
ax3 = fig.add_subplot(gs[2, 0])

# 绘制理论曲线 | Plot theoretical curves
phi_theory = np.linspace(0.2, 5, 100)
for n_val in [2.0, 2.5, 3.0, 3.5]:
    epsilon_theory = budyko_curve(phi_theory, n_val)
    ax3.plot(phi_theory, epsilon_theory, 'gray', alpha=0.3, linewidth=1)
    if n_val in [2.0, 3.5]:
        idx = 60
        ax3.text(phi_theory[idx], epsilon_theory[idx], f'n={n_val}', 
                fontsize=9, color='gray')

# 绘制能量限制线和水分限制线 | Plot energy and water limit lines
ax3.plot([0, 5], [1, 1], 'b--', linewidth=1.5, alpha=0.5, label='能量限制 | Energy limit')
ax3.plot([0, 5], [0, 1], 'r--', linewidth=1.5, alpha=0.5, label='水分限制 | Water limit')

# 绘制实际数据点 | Plot actual data points
ax3.scatter(phi_annual[ref_period], epsilon_annual[ref_period], 
           c='blue', s=50, alpha=0.6, label='基线期 | Baseline', edgecolors='k', linewidth=0.5)
ax3.scatter(phi_annual[comp_period], epsilon_annual[comp_period], 
           c='red', s=50, alpha=0.6, label='变化期 | Change', edgecolors='k', linewidth=0.5)

# 标注平均点 | Mark average points
ax3.scatter([phi_ref], [epsilon_ref], c='blue', s=200, marker='*', 
           edgecolors='k', linewidth=2, label='基线期平均 | Baseline mean', zorder=5)
phi_comp_avg = phi_annual[comp_period].mean()
epsilon_comp_avg = epsilon_annual[comp_period].mean()
ax3.scatter([phi_comp_avg], [epsilon_comp_avg], c='red', s=200, marker='*', 
           edgecolors='k', linewidth=2, label='变化期平均 | Change mean', zorder=5)

ax3.set_xlabel('干燥度指数 φ = PETe/P | Aridity Index', fontweight='bold', fontsize=11)
ax3.set_ylabel('蒸发比 ε = ET/P | Evaporative Index', fontweight='bold', fontsize=11)
ax3.set_title('Budyko 空间 | Budyko Space', fontweight='bold', fontsize=13)
ax3.set_xlim([0, 3])
ax3.set_ylim([0, 1])
ax3.legend(fontsize=9, loc='best')
ax3.grid(alpha=0.3)

# 4. 归因饼图 | Attribution pie chart
ax4 = fig.add_subplot(gs[2, 1])

contributions = [abs(delta_et_climate), abs(delta_et_landsurf)]
labels_pie = [f'气候变化\nClimate\n{abs(delta_et_climate):.1f} mm/yr\n({abs(delta_et_climate)/abs(delta_et_total)*100:.1f}%)',
              f'下垫面变化\nLand Surface\n{abs(delta_et_landsurf):.1f} mm/yr\n({abs(delta_et_landsurf)/abs(delta_et_total)*100:.1f}%)']
colors_pie = ['#FF6B6B', '#4ECDC4']
explode = (0.05, 0.05)

wedges, texts, autotexts = ax4.pie(contributions, labels=labels_pie, colors=colors_pie,
                                    autopct='', explode=explode,
                                    shadow=True, startangle=90,
                                    textprops={'fontsize': 11, 'fontweight': 'bold'})

ax4.set_title(f'ET 变化归因分解\nET Change Attribution\n(Total: {delta_et_total:.1f} mm/year)', 
             fontweight='bold', fontsize=13)

plt.savefig('figures/attribution_analysis_comprehensive.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 可视化完成！Visualization complete!")

## 🎯 核心要点总结 | Key Takeaways

### 1. 归因分析的三步法 | Three-Step Attribution Method

1. **识别变化 | Identify Changes**:
   - 定义参考期和对比期 | Define reference and comparison periods
   - 量化 ET、P、PETe 的变化 | Quantify changes in ET, P, PETe

2. **分解贡献 | Decompose Contributions**:
   - 气候贡献：固定 $n$，改变 $\phi$ | Climate: fix $n$, vary $\phi$
   - 下垫面贡献：固定 $\phi$，改变 $n$ | Land surface: fix $\phi$, vary $n$

3. **验证守恒 | Verify Conservation**:
   - 确保 $\Delta ET_{\text{重构}} \approx \Delta ET_{\text{观测}}$ | Ensure reconstructed ≈ observed

---

### 2. Budyko 参数 $n$ 的物理意义 | Physical Meaning of Budyko Parameter $n$

| $n$ 值 | 物理特征 | 典型下垫面 |
|--------|---------|----------|
| **低 (1.5-2.0)** | 易径流 | 城市、裸地 |
| Low | High runoff | Urban, barren |
| **中 (2.0-3.0)** | 平衡 | 草地、农田 |
| Medium | Balanced | Grassland, cropland |
| **高 (3.0-4.0)** | 易蒸发 | 森林、湿地 |
| High | High evaporation | Forest, wetland |

---

### 3. 实际应用案例 | Real-World Applications

| 区域 | 主要发现 | 归因结论 |
|------|---------|--------|
| **黄河流域** | 径流减少 30% | 气候 40% + 植树 60% |
| Yellow River | Runoff ↓30% | Climate 40% + Afforestation 60% |
| **亚马逊雨林** | ET 增加 | 气候变暖主导 |
| Amazon | ET ↑ | Climate warming dominant |
| **澳大利亚** | 干旱加剧 | 降水减少主导 |
| Australia | Drought intensified | Precipitation reduction dominant |

---

### 4. 归因分析的不确定性 | Uncertainties in Attribution

1. **数据不确定性 | Data Uncertainty**:
   - 观测误差（降水、气温）
   - 模型参数误差

2. **方法不确定性 | Methodological Uncertainty**:
   - Budyko 参数化方案选择
   - 时间窗口选择

3. **交互作用 | Interactions**:
   - 气候和下垫面变化可能相互影响
   - 非线性效应

---

## 💡 思考题 | Discussion Questions

1. **如果一个流域 $n$ 参数从 2.5 增加到 3.0，可能是什么原因？**
   - If a catchment's $n$ parameter increases from 2.5 to 3.0, what might be the reason?
   - 提示：考虑植被覆盖、水库建设、灌溉
   - Hint: Consider vegetation cover, reservoir construction, irrigation

2. **为什么植树造林会增加蒸散发？这对下游径流有什么影响？**
   - Why does afforestation increase evapotranspiration? What impact does this have on downstream runoff?
   - 提示：树木的蒸腾作用 vs 草地
   - Hint: Tree transpiration vs grassland

3. **在气候变化背景下，如何平衡生态修复和水资源管理？**
   - Under climate change, how to balance ecological restoration and water resource management?
   - 提示：黄河流域的"用水红线"
   - Hint: Yellow River Basin's "water use red line"

---

## 📚 延伸阅读 | Further Reading

1. **Zhou & Yu (2025)**: Land-atmosphere interactions exacerbate concurrent soil moisture drought and atmospheric aridity. *Nature Climate Change* (accepted).

2. **Fu (1981)**: On the calculation of the evaporation from land surface. *Scientia Atmospherica Sinica*.

3. **Budyko (1974)**: *Climate and Life*. Academic Press.

---

## 🎓 恭喜！| Congratulations!

你已经完成了 PET-CR 教程系列！现在你可以：

You've completed the PET-CR tutorial series! Now you can:

- ✅ 理解蒸散发的物理基础 | Understand the physical basis of ET
- ✅ 掌握互补关系理论 | Master complementary relationship theory
- ✅ 进行水文归因分析 | Conduct hydrological attribution analysis
- ✅ 用 Python 处理实际数据 | Process real data with Python

**下一步 | Next Steps**:
- 探索 `examples/` 目录中的高级案例
- 阅读 `docs/THEORY.md` 了解更多理论细节
- 使用自己的数据进行分析！

---

**作者 | Author**: PET-CR Development Team  
**版本 | Version**: 1.0  
**日期 | Date**: 2025-12-04